# 05. Colab Supervised Fine-Tuning - LocalFit v1

이 노트북의 목적은 `04_build_final_culane_dataset.ipynb`에서 만든 `map_culane_localfit_train_field1_field2_v1` 데이터셋을 Colab의 CLRKDNet 공식 코드에 올려서, `ResNet18_CULane.pth`를 우리 맵 도메인에 맞게 supervised fine-tuning하는 것이다.

이번 노트북에서 특히 확인할 것:

1. 데이터셋이 공식 CULane reader가 읽는 구조인지 확인한다.
2. 학습 입력이 실제로 BGR float `[0, 1]` 범위인지 확인한다.
3. 공식 `configs/ResNet18_CULane.py`에서 필요한 값만 패치한다.
4. smoke 1 epoch로 전체 파이프라인을 먼저 통과시킨다.
5. full fine-tuning 결과 `.pth`와 로그를 Drive에 보관한다.

ONNX export와 Pi runtime 검증은 다음 노트북에서 한다. 여기서는 의도적으로 `.pth` 학습 결과까지만 만든다.


## 0. Colab 입력 파일

아래 파일 2개를 `DRIVE_DIR`에 올려둔다.

- `map_culane_localfit_train_field1_field2_v1.tar.gz`
- `ResNet18_CULane.pth`

`map_culane_localfit_train_field1_field2_v1.tar.gz`는 로컬 `20_shared_assets/dataset/lane`에 있는 최종 데이터셋 폴더를 압축한 파일이다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time, tarfile, textwrap

DRIVE_DIR = Path('/content/drive/MyDrive/Colab Notebooks/26-1학기_임베디드인공지능시스템최적화/06_CLRKDNet_Fine-Tuning_v4')
REPO_DIR = Path('/content/CLRKDNet')
DATA_ROOT = REPO_DIR / 'data'

DATASET_NAME = 'map_culane_localfit_train_field1_field2_v1'
DATASET_TAR = DRIVE_DIR / f'{DATASET_NAME}.tar.gz'
DATASET_DIR = DATA_ROOT / DATASET_NAME

PRETRAINED_NAME = 'ResNet18_CULane.pth'
CKPT_SRC = DRIVE_DIR / PRETRAINED_NAME
CKPT_DST = REPO_DIR / PRETRAINED_NAME

OUT_DRIVE = DRIVE_DIR / 'outputs_localfit_v1'
OUT_DRIVE.mkdir(parents=True, exist_ok=True)

RUN_PREFIX = 'MapLane_LocalFit_Field12_v1'

print('DRIVE_DIR:', DRIVE_DIR)
print('dataset tar:', DATASET_TAR, 'exists=', DATASET_TAR.exists())
print('pretrained:', CKPT_SRC, 'exists=', CKPT_SRC.exists())
print('output:', OUT_DRIVE)
assert DATASET_TAR.exists(), f'missing dataset tar: {DATASET_TAR}'
assert CKPT_SRC.exists(), f'missing checkpoint: {CKPT_SRC}'


## 1. 공식 Repo 설치와 호환성 패치

학습 로직은 공식 CLRKDNet repo를 그대로 사용한다. Colab 최신 환경에서 깨지는 부분만 최소 호환성 패치한다.

주의할 점은 validation 단계의 NMS다. 공식 repo는 CUDA extension 기반 NMS를 쓰는데, Colab 환경에서 extension import가 실패할 수 있다. 이 노트북은 그 경우 validation을 계속 진행하기 위한 fallback을 둔다. 학습 loss 계산 자체는 이 fallback에 의존하지 않는다.


In [ ]:
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/weiqingq/CLRKDNet.git', str(REPO_DIR)], check=True)
else:
    print('repo already exists:', REPO_DIR)

print('repo files:', len(list(REPO_DIR.rglob('*'))))


In [ ]:
# Colab runtime compatibility packages.
# Keep Colab's core binary stack (torch/torchvision/numpy/pillow/opencv) intact.
# Only install small packages that are usually missing from the official CLRKDNet repo requirements.
import importlib
import numpy as np

if not hasattr(np, 'sctypes'):
    np.sctypes = {
        'float': [np.float16, np.float32, np.float64],
        'int': [np.int8, np.int16, np.int32, np.int64],
        'uint': [np.uint8, np.uint16, np.uint32, np.uint64],
        'complex': [np.complex64, np.complex128],
        'others': [np.bool_, np.object_, np.bytes_, np.str_],
    }

checks = [
    ('addict', 'addict'),
    ('yapf', 'yapf==0.40.1'),
    ('pathspec', 'pathspec'),
    ('timm', 'timm'),
    ('pytorch_warmup', 'pytorch_warmup'),
    ('ptflops', 'ptflops'),
    ('imgaug', 'imgaug'),
    ('shapely', 'shapely'),
    ('p_tqdm', 'p_tqdm'),
]
missing = []
for module_name, pip_name in checks:
    try:
        importlib.import_module(module_name)
    except Exception:
        missing.append(pip_name)

if missing:
    print('installing missing packages:', missing)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
else:
    print('all optional packages already available')

print('numpy:', np.__version__)
try:
    import cv2
    print('cv2:', cv2.__version__)
except Exception as e:
    print('cv2 import failed:', repr(e))


In [ ]:
def write_text(path: Path, text: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(text).lstrip(), encoding='utf-8')

# This experiment only uses CULane. Avoid importing TuSimple because tusimple.py imports torchvision,
# which is unnecessary here and can trip Colab binary compatibility issues.
write_text(REPO_DIR / 'clrkd' / 'datasets' / '__init__.py', '''
from .registry import build_dataset, build_dataloader
from .culane import CULane
from .process import *
''')

# base_dataset.py imports torchvision but does not use it in the CULane path.
# Remove the import to avoid Colab torchvision/Pillow binary compatibility failures.
base_dataset_py = REPO_DIR / 'clrkd' / 'datasets' / 'base_dataset.py'
base_text = base_dataset_py.read_text(encoding='utf-8')
base_text = base_text.replace('import torchvision\n', '')
base_dataset_py.write_text(base_text, encoding='utf-8')

# Minimal MMCV shim for this repo's imports.
write_text(REPO_DIR / 'mmcv' / '__init__.py', '''
__version__ = 'local-shim'

def jit(*jit_args, **jit_kwargs):
    def decorator(func):
        return func
    if len(jit_args) == 1 and callable(jit_args[0]) and not jit_kwargs:
        return jit_args[0]
    return decorator

def load(filename, *args, **kwargs):
    import json
    with open(filename, 'r') as f:
        return json.load(f)

def dump(obj, file=None, file_format=None, *args, **kwargs):
    import json
    text = json.dumps(obj, indent=2)
    if file is None:
        return text
    with open(file, 'w') as f:
        f.write(text)
''')
write_text(REPO_DIR / 'mmcv' / 'cnn' / '__init__.py', '''
import torch.nn as nn

class ConvModule(nn.Sequential):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0,
                 dilation=1, groups=1, bias='auto', conv_cfg=None, norm_cfg=None,
                 act_cfg=dict(type='ReLU'), inplace=True, **kwargs):
        layers = []
        use_bias = bias if isinstance(bias, bool) else (norm_cfg is None)
        layers.append(nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride,
                                padding=padding, dilation=dilation, groups=groups, bias=use_bias))
        if norm_cfg is not None:
            layers.append(nn.BatchNorm2d(out_channels))
        if act_cfg is not None:
            layers.append(nn.ReLU(inplace=inplace))
        super().__init__(*layers)
''')
write_text(REPO_DIR / 'mmcv' / 'parallel' / '__init__.py', '''
import torch
from torch.utils.data._utils.collate import default_collate

class DataContainer:
    def __init__(self, data, stack=False, padding_value=0, cpu_only=False, pad_dims=2):
        self.data = data
        self.stack = stack
        self.padding_value = padding_value
        self.cpu_only = cpu_only
        self.pad_dims = pad_dims

def collate(batch, samples_per_gpu=1):
    if not batch:
        return batch
    elem = batch[0]
    if isinstance(elem, DataContainer):
        return [b.data for b in batch] if elem.cpu_only else default_collate([b.data for b in batch])
    if isinstance(elem, dict):
        return {key: collate([d[key] for d in batch], samples_per_gpu) for key in elem}
    if isinstance(elem, (list, tuple)):
        transposed = list(zip(*batch))
        return [collate(samples, samples_per_gpu) for samples in transposed]
    try:
        return default_collate(batch)
    except Exception:
        return batch

class MMDataParallel(torch.nn.DataParallel):
    def __init__(self, module, device_ids=None, dim=0):
        if device_ids is None:
            device_ids = [0] if torch.cuda.is_available() else []
        super().__init__(module, device_ids=device_ids, dim=dim)
''')

# Python 3.10+ collections compatibility.
for path in REPO_DIR.rglob('*.py'):
    text = path.read_text(encoding='utf-8')
    new = text.replace('collections.Iterable', 'collections.abc.Iterable')
    if new != text:
        if 'import collections.abc' not in new:
            new = new.replace('import collections\n', 'import collections\nimport collections.abc\n')
        path.write_text(new, encoding='utf-8')

# NumPy 2 removed np.sctypes; keep imgaug safe if the runtime upgrades NumPy later.
write_text(REPO_DIR / 'sitecustomize.py', '''
import numpy as np
if not hasattr(np, 'sctypes'):
    np.sctypes = {
        'float': [np.float16, np.float32, np.float64],
        'int': [np.int8, np.int16, np.int32, np.int64],
        'uint': [np.uint8, np.uint16, np.uint32, np.uint64],
        'complex': [np.complex64, np.complex128],
        'others': [np.bool_, np.object_, np.bytes_, np.str_],
    }
''')

# Check the shim immediately so import-time failures are caught here, not during training.
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
import importlib, mmcv
mmcv = importlib.reload(mmcv)
for name in ['jit', 'load', 'dump']:
    assert hasattr(mmcv, name), f'mmcv shim missing {name}'
print('compatibility patches written and verified')


In [ ]:
# Validation NMS fallback.
# Official training is unchanged. This only prevents validation from crashing when the CUDA NMS op is unavailable.
write_text(REPO_DIR / 'clrkd' / 'ops' / 'nms.py', '''
import torch

def _mean_abs_lane_distance(a, b):
    # nms input after CLRHead preprocessing: index 5: contains absolute x coordinates.
    ax = a[5:]
    bx = b[5:]
    valid = torch.isfinite(ax) & torch.isfinite(bx) & (ax >= 0) & (bx >= 0)
    if int(valid.sum().item()) < 2:
        return torch.tensor(float('inf'), device=a.device)
    return torch.mean(torch.abs(ax[valid] - bx[valid]))

def nms(boxes, scores, overlap=50, top_k=4):
    if boxes is None or scores is None or scores.numel() == 0:
        keep = torch.empty((0,), dtype=torch.long, device=scores.device if scores is not None else 'cpu')
        return keep, 0, None

    order = torch.argsort(scores, descending=True)
    kept = []
    for idx in order:
        if len(kept) >= int(top_k):
            break
        duplicate = False
        for kept_idx in kept:
            if _mean_abs_lane_distance(boxes[idx], boxes[kept_idx]) <= overlap:
                duplicate = True
                break
        if not duplicate:
            kept.append(idx)

    if not kept:
        keep = torch.empty((0,), dtype=torch.long, device=scores.device)
    else:
        keep = torch.stack(kept).long()
    return keep, int(keep.numel()), None
''')

# Patch CULane metric geometry to use our raw image size, not the hard-coded CULane 1640x590 assumptions.
culane_py = REPO_DIR / 'clrkd' / 'datasets' / 'culane.py'
text = culane_py.read_text(encoding='utf-8')

ys_target = 'ys = np.arange(self.cfg.cut_height, self.cfg.ori_img_h, 8) / self.cfg.ori_img_h'
ys_old_patterns = [
    'ys = np.arange(270, 590, 8) / self.cfg.ori_img_h',
    'ys = np.arange(270, 590, 8) / 590',
]
if ys_target not in text:
    ys_applied = False
    for old in ys_old_patterns:
        if old in text:
            text = text.replace(old, ys_target)
            ys_applied = True
    assert ys_applied, 'culane.py sample-y geometry patch did not find an expected source pattern'
assert ys_target in text, 'culane.py sample-y geometry patch did not apply'
assert 'np.arange(270, 590, 8)' not in text, 'stale CULane sample-y range remains in culane.py'

call_target = 'official=True, img_shape=(self.cfg.ori_img_h, self.cfg.ori_img_w, 3))'
if call_target not in text:
    old_call_count = text.count('official=True)')
    assert old_call_count >= 2, f'expected at least 2 official=True calls, got {old_call_count}'
    text = text.replace('official=True)', call_target)
assert text.count(call_target) >= 2, 'culane.py eval_predictions img_shape patch did not apply to both calls'
culane_py.write_text(text, encoding='utf-8')

metric_py = REPO_DIR / 'clrkd' / 'utils' / 'culane_metric.py'
text = metric_py.read_text(encoding='utf-8')
signature_target = 'sequential=False,\n                     img_shape=(590, 1640, 3)):'
if signature_target not in text:
    assert 'sequential=False):' in text, 'culane_metric.py signature source pattern not found'
    text = text.replace('sequential=False):', signature_target)
assert signature_target in text, 'culane_metric.py img_shape signature patch did not apply'

if 'img_shape = (590, 1640, 3)' in text:
    text = text.replace(
        'img_shape = (590, 1640, 3)',
        '# img_shape is provided by dataset config for project-map data'
    )
assert 'img_shape = (590, 1640, 3)' not in text, 'stale hard-coded metric img_shape remains'
metric_py.write_text(text, encoding='utf-8')

print('validation geometry patch applied and verified')


## 2. 데이터셋 압축 해제와 구조 확인

이 단계가 통과해야 학습으로 넘어간다. `train_gt.txt`, `val.txt`, `test.txt` count가 로컬 04 노트북 결과와 일치해야 한다.


In [ ]:
DATA_ROOT.mkdir(parents=True, exist_ok=True)

if DATASET_DIR.exists():
    print('dataset already extracted:', DATASET_DIR)
else:
    print('extracting:', DATASET_TAR)
    with tarfile.open(DATASET_TAR, 'r:gz') as tf:
        tf.extractall(DATA_ROOT)
    print('extracted:', DATASET_DIR)

shutil.copy2(CKPT_SRC, CKPT_DST)

# Avoid stale official CULane cache from a previous dataset/config.
cache_dir = REPO_DIR / 'cache'
if cache_dir.exists():
    shutil.rmtree(cache_dir)
    print('removed stale cache:', cache_dir)

assert DATASET_DIR.exists(), DATASET_DIR
assert CKPT_DST.exists(), CKPT_DST
print('checkpoint copied:', CKPT_DST)


In [ ]:
def load_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def count_lines(path):
    path = Path(path)
    return sum(1 for _ in path.open('r', encoding='utf-8')) if path.exists() else None

summary = load_json(DATASET_DIR / 'build_summary.json')
validation = load_json(DATASET_DIR / 'validation_summary.json')
print('build_summary:')
print(json.dumps(summary, ensure_ascii=False, indent=2)[:3000])
print('\nvalidation_summary:')
print(json.dumps(validation, ensure_ascii=False, indent=2))

for rel in ['list/train_gt.txt', 'list/val.txt', 'list/test.txt', 'list/test_split/test0_normal.txt']:
    print(rel, count_lines(DATASET_DIR / rel))


In [ ]:
EXPECTED_TRAIN = 6894
EXPECTED_VAL = 1722
EXPECTED_TEST = 1722

train_count = count_lines(DATASET_DIR / 'list/train_gt.txt')
val_count = count_lines(DATASET_DIR / 'list/val.txt')
test_count = count_lines(DATASET_DIR / 'list/test.txt')
assert train_count == EXPECTED_TRAIN, (train_count, EXPECTED_TRAIN)
assert val_count == EXPECTED_VAL, (val_count, EXPECTED_VAL)
assert test_count == EXPECTED_TEST, (test_count, EXPECTED_TEST)
assert validation['issue_count'] == 0, validation.get('issues_sample')
print('STOP CHECK A passed: dataset list counts and validation summary are clean.')


## 3-0. 실험 설계 결정 근거

이 노트북은 `localfit v1` 데이터셋으로 **공식 ResNet18_CULane supervised fine-tuning baseline**을 만드는 것이 목적이다. 따라서 architecture, loss weight, scheduler, 공식 train transform 구조는 우선 그대로 둔다. 여기서 값을 많이 바꾸면, 이전 v1/v2 문제의 원인이 입력 정규화였는지, 라벨 정책이었는지, augmentation이었는지 분리하기 어려워진다.

결정 근거:

- `epochs=12`: 이전 field1+2 fine-tuning v1과 비교 가능한 full run 길이를 유지한다. 데이터셋이 6,894 train frame이라 12 epoch는 약 10k iteration이다.
- `batch_size=8`: Colab T4 메모리에서 안정적으로 돌리기 위한 값이다.
- `lr=1e-4`: 공식 base `6e-4`보다 작게 둔 보수적 fine-tuning lr이다. batch 24에서 8로 줄인 선형 scaling 기준은 약 `2e-4`지만, local-fit pseudo label의 노이즈를 고려해 더 낮게 시작한다.
- `conf_threshold=0.35`: validation에서 초기 under-confidence로 metric이 0에 붙는 것을 피하기 위한 약간 낮춘 값이다. 최종 Pi runtime threshold는 다음 export/runtime 노트북에서 다시 sweep한다.
- `augmentation`: 이번 baseline은 공식 CULane augmentation을 유지한다. 단, 우리 도메인은 노란색 차선이 강한 단서라 `ChannelShuffle`, `Hue/Saturation jitter`는 성능이 안 좋을 때 가장 먼저 ablation할 후보로 기록한다.
- `NMS fallback`: Colab에서 공식 CUDA NMS extension을 쓰기 어려워 validation용 fallback을 둔다. 학습 loss에는 영향이 없지만, `best.pth` 선정 metric에는 영향을 줄 수 있다. 따라서 full run 이후에는 `best.pth`만 맹신하지 말고 저장된 epoch checkpoint 전체를 후속 분석 노트북에서 overlay/holdout 기준으로 다시 비교한다.


## 3. 학습 Config 생성

공식 `configs/ResNet18_CULane.py`를 기반으로 아래 항목만 바꾼다.

- raw image geometry: `1296x972`, `cut_height=445`, model input `800x320`
- dataset path: `./data/map_culane_localfit_train_field1_field2_v1`
- schedule: smoke 1 epoch, full 12 epoch
- backbone pretrained flag: `False`, 실제 가중치는 `--finetune_from ResNet18_CULane.pth`로 로드

여기서 NMS threshold는 공식 기본 흐름의 validation 파라미터다. 최종 Pi runtime의 후처리는 ONNX export 이후 다시 별도 검증한다.


In [ ]:
BASE_CONFIG = REPO_DIR / 'configs' / 'ResNet18_CULane.py'
CONFIG_SMOKE = REPO_DIR / 'configs' / f'{RUN_PREFIX}_ResNet18_smoke.py'
CONFIG_FULL = REPO_DIR / 'configs' / f'{RUN_PREFIX}_ResNet18_full.py'

BATCH_SIZE = 8
LR = 1.0e-4
SMOKE_EPOCHS = 1
FULL_EPOCHS = 12

# Baseline policy: keep official CULane augmentation.
# If this run underperforms, the next controlled ablation should set this to 'domain_safe'
# to remove ChannelShuffle and reduce Hue/Saturation jitter for the yellow-lane indoor map.
AUG_POLICY = 'official'  # choices: 'official', 'domain_safe'
TRAIN_ROWS = train_count
TOTAL_ITER_SMOKE = (TRAIN_ROWS // BATCH_SIZE) * SMOKE_EPOCHS
TOTAL_ITER_FULL = (TRAIN_ROWS // BATCH_SIZE) * FULL_EPOCHS

print('BASE_CONFIG:', BASE_CONFIG)
print('CONFIG_SMOKE:', CONFIG_SMOKE)
print('CONFIG_FULL:', CONFIG_FULL)
print('TRAIN_ROWS:', TRAIN_ROWS)
print('TOTAL_ITER_SMOKE:', TOTAL_ITER_SMOKE)
print('TOTAL_ITER_FULL:', TOTAL_ITER_FULL)
print('AUG_POLICY:', AUG_POLICY)


In [ ]:
def replace_assignment(text, name, value):
    import re
    pattern = rf'^{name}\s*=.*$'
    replacement = f'{name} = {value}'
    new, n = re.subn(pattern, replacement, text, flags=re.MULTILINE)
    if n == 0:
        new += f'\n{replacement}\n'
    return new

def patch_config(out_path: Path, *, epochs: int, total_iter: int, work_dir: str, eval_ep: int, save_ep: int, lr: float):
    text = BASE_CONFIG.read_text(encoding='utf-8')
    header = f'''# Auto-generated from official configs/ResNet18_CULane.py
# Patched for project-map geometry, localfit v1 dataset, schedule, and finetune-safe backbone init.
# TRAIN_ROWS={TRAIN_ROWS}, BATCH_SIZE={BATCH_SIZE}, LR={lr}
\n'''
    text = header + text
    replacements = {
        'sample_y': 'range(971, 444, -20)',
        'ori_img_w': '1296',
        'ori_img_h': '972',
        'img_w': '800',
        'img_h': '320',
        'cut_height': '445',
        'epochs': str(epochs),
        'batch_size': str(BATCH_SIZE),
        'eval_ep': str(eval_ep),
        'save_ep': str(save_ep),
        'total_iter': str(total_iter),
        'work_dirs': repr(f'work_dirs/{work_dir}'),
        'dataset_path': repr(f'./data/{DATASET_NAME}'),
        'workers': '2',
        'log_interval': '20',
        'diff_path': 'None',
    }
    for k, v in replacements.items():
        text = replace_assignment(text, k, v)

    text = text.replace("optimizer = dict(type='AdamW', lr=0.6e-3)", f"optimizer = dict(type='AdamW', lr={lr})")
    text = text.replace("optimizer = dict(type='AdamW', lr=6e-4)", f"optimizer = dict(type='AdamW', lr={lr})")
    text = text.replace("pretrained=True", "pretrained=False")
    text = text.replace("split='test'", "split='val'", 1)
    text = text.replace("# seed = 0", "seed = 0")
    text = replace_assignment(text, 'test_parameters', "dict(conf_threshold=0.35, nms_thres=50, nms_topk=max_lanes)")

    if AUG_POLICY == 'domain_safe':
        # Controlled ablation option for the yellow-lane indoor map. Keep geometry/loss/architecture fixed.
        text = text.replace("            dict(name='ChannelShuffle', parameters=dict(p=1.0), p=0.1),\n", '')
        text = text.replace("            dict(name='AddToHueAndSaturation',\n                 parameters=dict(value=(-10, 10)),\n                 p=0.7),",
                            "            dict(name='AddToHueAndSaturation',\n                 parameters=dict(value=(-5, 5)),\n                 p=0.3),")
    elif AUG_POLICY != 'official':
        raise ValueError(f'unknown AUG_POLICY: {AUG_POLICY}')

    out_path.write_text(text, encoding='utf-8')
    return out_path

patch_config(CONFIG_SMOKE, epochs=SMOKE_EPOCHS, total_iter=TOTAL_ITER_SMOKE,
             work_dir=f'{RUN_PREFIX}_smoke', eval_ep=1, save_ep=1, lr=LR)
patch_config(CONFIG_FULL, epochs=FULL_EPOCHS, total_iter=TOTAL_ITER_FULL,
             work_dir=f'{RUN_PREFIX}_full', eval_ep=2, save_ep=2, lr=LR)

preview = CONFIG_FULL.read_text(encoding='utf-8').splitlines()[:80]
print('\n'.join(preview))


## 4. Stop Check B - 학습 입력 분포 확인

이전 실험의 가장 큰 문제는 학습 입력과 runtime 입력 scale이 달랐다는 점이었다. 따라서 이번에는 학습 dataloader가 모델에 넣는 이미지 tensor의 dtype/range/order를 직접 확인한다.

통과 기준:

- shape: 대략 `[3, 320, 800]`
- dtype: `float32`
- min/max: `[0, 1]` 범위

다음 ONNX export 노트북은 반드시 이 입력 분포와 같은 preprocess를 써야 한다.


In [ ]:
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

from clrkd.utils.config import Config

cfg = Config.fromfile(str(CONFIG_SMOKE))

# The registry is populated by importing dataset modules. In an interactive Colab
# kernel, a partial/failed import can leave the registry empty, so reset and import explicitly.
for module_name in list(sys.modules):
    if module_name == 'clrkd.datasets' or module_name.startswith('clrkd.datasets.'):
        del sys.modules[module_name]

import clrkd.datasets  # imports CULane, TuSimple, and process modules through __init__.py
import clrkd.datasets.culane
import clrkd.datasets.process
from clrkd.datasets import build_dataset
from clrkd.datasets.registry import DATASETS, PROCESS

print('dataset package:', clrkd.datasets.__file__)
print('registered datasets:', sorted(DATASETS.module_dict.keys()))
print('registered processes sample:', sorted(PROCESS.module_dict.keys())[:12])
assert 'CULane' in DATASETS.module_dict, DATASETS
assert 'GenerateLaneLine' in PROCESS.module_dict, PROCESS
assert 'ToTensor' in PROCESS.module_dict, PROCESS

train_dataset = build_dataset(cfg.dataset.train, cfg)

sample = train_dataset[0]
img = sample.get('img') if isinstance(sample, dict) else None
if hasattr(img, 'data'):
    img = img.data

import torch
if isinstance(img, torch.Tensor):
    img_tensor = img
else:
    img_tensor = torch.as_tensor(img)

print('sample keys:', list(sample.keys()) if isinstance(sample, dict) else type(sample))
print('img shape:', tuple(img_tensor.shape))
print('img dtype:', img_tensor.dtype)
print('img min/max/mean:', float(img_tensor.min()), float(img_tensor.max()), float(img_tensor.float().mean()))

assert img_tensor.ndim == 3, tuple(img_tensor.shape)
assert img_tensor.shape[-2:] == (320, 800), tuple(img_tensor.shape)
assert float(img_tensor.max()) <= 1.01, 'training tensor is not [0,1]; export/runtime must match this check'
assert float(img_tensor.min()) >= -0.01, 'unexpected negative image values'
print('STOP CHECK B passed: official training pipeline feeds BGR float in [0,1].')


## 5. Stop Check C0 - 공식 모델 import/build 사전 검증

Smoke 학습을 시작하기 전에 공식 모델 import와 `build_net(cfg)`를 먼저 실행한다. 이 셀은 학습을 돌리기 전에 다음 문제를 조기에 잡기 위한 안전장치다.

- `mmcv.jit` 누락
- model/head/backbone registry 누락
- `CLRHead`, `Detector`, `ResNetWrapper` import 실패
- 불필요한 `torchvision` import 재유입


In [ ]:
import importlib
import mmcv
mmcv = importlib.reload(mmcv)
for name in ['jit', 'load', 'dump']:
    assert hasattr(mmcv, name), f'mmcv shim missing {name}'

import clrkd.models
from clrkd.models.registry import BACKBONES, NECKS, HEADS, NETS, build_net

print('registered nets:', sorted(NETS.module_dict.keys()))
print('registered backbones sample:', sorted(BACKBONES.module_dict.keys())[:8])
print('registered necks:', sorted(NECKS.module_dict.keys()))
print('registered heads:', sorted(HEADS.module_dict.keys()))
assert 'Detector' in NETS.module_dict, NETS
assert 'ResNetWrapper' in BACKBONES.module_dict, BACKBONES
assert 'Aggregator' in NECKS.module_dict, NECKS
assert 'CLRHead' in HEADS.module_dict, HEADS

model_probe = build_net(cfg)
param_count = sum(p.numel() for p in model_probe.parameters())
print('model:', type(model_probe).__name__)
print('params:', param_count)
del model_probe
print('STOP CHECK C0 passed: official model import/build path is valid.')


## 5-1. Stop Check C1 - subprocess 환경 사전 검증

Smoke 학습은 별도 `python main.py ...` 프로세스에서 실행된다. 현재 노트북 커널에서 import가 성공해도, subprocess에서 `sitecustomize`, repo patch, `PYTHONPATH`가 다르게 보이면 실패할 수 있다.

따라서 smoke 학습 전에 같은 subprocess 방식으로 공식 config 로드와 모델 build를 먼저 확인한다.


In [ ]:
preflight_code = f"""
import os, sys, importlib
sys.path.insert(0, {str(REPO_DIR)!r})
os.chdir({str(REPO_DIR)!r})
import numpy as np
if not hasattr(np, 'sctypes'):
    np.sctypes = {{
        'float': [np.float16, np.float32, np.float64],
        'int': [np.int8, np.int16, np.int32, np.int64],
        'uint': [np.uint8, np.uint16, np.uint32, np.uint64],
        'complex': [np.complex64, np.complex128],
        'others': [np.bool_, np.object_, np.bytes_, np.str_],
    }}
import mmcv
for name in ['jit', 'load', 'dump']:
    assert hasattr(mmcv, name), f'mmcv shim missing {{name}}'
import clrkd.datasets
import clrkd.models
from clrkd.utils.config import Config
from clrkd.datasets.registry import DATASETS, PROCESS
from clrkd.models.registry import BACKBONES, NECKS, HEADS, NETS, build_net
cfg = Config.fromfile({str(CONFIG_SMOKE)!r})
assert 'CULane' in DATASETS.module_dict, DATASETS
assert 'Detector' in NETS.module_dict, NETS
assert 'CLRHead' in HEADS.module_dict, HEADS
model = build_net(cfg)
print('subprocess numpy', np.__version__)
print('subprocess model', type(model).__name__, 'params', sum(p.numel() for p in model.parameters()))
print('STOP CHECK C1 passed: subprocess import/build path is valid.')
"""
env = os.environ.copy()
env['PYTHONPATH'] = str(REPO_DIR) + (':' + env['PYTHONPATH'] if env.get('PYTHONPATH') else '')
proc = subprocess.run(
    ['python', '-c', preflight_code],
    cwd=REPO_DIR,
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(proc.stdout)
assert proc.returncode == 0, proc.stdout[-12000:]


## 5. Smoke 학습

1 epoch smoke run은 데이터셋, config, dataloader, loss, validation, checkpoint 저장까지 한 번에 통과하는지 확인하기 위한 단계다.

Smoke metric 자체는 최종 성능 판단용이 아니다. 여기서 에러가 없어야 full run으로 넘어간다.


In [ ]:
RUN_SMOKE = True
RUN_FULL = True  # smoke만 확인하려면 False로 바꾼다.


def latest_run_dir(work_name: str):
    root = REPO_DIR / 'work_dirs' / work_name
    if not root.exists():
        return None
    candidates = [p for p in root.iterdir() if p.is_dir()]
    return max(candidates, key=lambda p: p.stat().st_mtime) if candidates else None


def tail_log(run_dir: Path, n=80):
    log_path = run_dir / 'log.txt'
    if not log_path.exists():
        return ''
    lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
    return '\n'.join(lines[-n:])


def should_live_print(line: str) -> bool:
    # Colab UI 안정성을 위해 핵심 줄만 출력한다.
    # 전체 stdout은 /content/CLRKDNet/runtime_logs 에 먼저 저장하고, 실행 종료 후 Drive로 복사한다.
    if ' - INFO - epoch:' in line:
        # log_interval=20이라 모든 학습 로그를 출력하면 셀 출력이 길어진다.
        # 100 step 단위와 epoch 시작/끝 근처만 보여준다.
        import re
        m = re.search(r'step:\s*(\d+)', line)
        if not m:
            return True
        step = int(m.group(1))
        return step <= 5 or step % 100 == 1 or step % 862 in (0, 1)
    keep_tokens = [
        'Build train loader',
        'Start training',
        'Number of images loaded',
        'Generating prediction output',
        'Calculating metric for List:',
        'iou thr: 0.50',
        'mean result',
        'metric:',
        'Best metric',
        'Traceback',
        'Error',
        'Exception',
    ]
    return any(token in line for token in keep_tokens)


def read_tail(path: Path, chars=12000):
    if not path.exists():
        return ''
    text = path.read_text(encoding='utf-8', errors='replace')
    return text[-chars:]


def copy_debug_log(local_path: Path, drive_path: Path):
    try:
        drive_path.parent.mkdir(parents=True, exist_ok=True)
        drive_path.write_text(local_path.read_text(encoding='utf-8', errors='replace'), encoding='utf-8', errors='replace')
        return True
    except Exception as e:
        print('debug log copy failed:', repr(e))
        return False


def mirror_checkpoints_to_drive(work_name: str):
    # 런타임이 중간에 죽어도 best.pth/epoch checkpoint를 잃지 않도록 가벼운 파일만 Drive에 복사한다.
    import shutil
    run_dir = latest_run_dir(work_name)
    if run_dir is None:
        return None
    mirror_dir = OUT_DRIVE / 'checkpoint_mirror' / work_name / run_dir.name
    mirror_dir.mkdir(parents=True, exist_ok=True)

    copied = []
    log_path = run_dir / 'log.txt'
    if log_path.exists():
        shutil.copy2(log_path, mirror_dir / 'log.txt')
        copied.append('log.txt')

    ckpt_dir = run_dir / 'ckpt'
    if ckpt_dir.exists():
        out_ckpt = mirror_dir / 'ckpt'
        out_ckpt.mkdir(parents=True, exist_ok=True)
        for src in sorted(ckpt_dir.glob('*.pth')):
            dst = out_ckpt / src.name
            if (not dst.exists()) or src.stat().st_size != dst.stat().st_size:
                shutil.copy2(src, dst)
            copied.append(f'ckpt/{src.name}')
    if copied:
        print(f'[mirror] {work_name}: {copied[-6:]} -> {mirror_dir}', flush=True)
    return mirror_dir


def run_training(config_path: Path, work_name: str, enabled=True):
    if not enabled:
        print('[skip]', work_name)
        return None
    cmd = [
        'python', 'main.py', str(config_path),
        '--gpus', '0',
        '--finetune_from', str(CKPT_DST),
        '--work_dirs', f'work_dirs/{work_name}',
    ]
    print('[run]', work_name)
    print('command:', ' '.join(cmd), flush=True)
    start = time.time()
    env = os.environ.copy()
    env['PYTHONPATH'] = str(REPO_DIR) + (':' + env['PYTHONPATH'] if env.get('PYTHONPATH') else '')
    env['PYTHONUNBUFFERED'] = '1'

    local_debug_dir = REPO_DIR / 'runtime_logs'
    local_debug_dir.mkdir(parents=True, exist_ok=True)
    local_debug_path = local_debug_dir / f'{work_name}_subprocess_output.txt'
    drive_debug_path = OUT_DRIVE / 'debug_logs' / f'{work_name}_subprocess_output.txt'

    proc = subprocess.Popen(
        cmd,
        cwd=REPO_DIR,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    assert proc.stdout is not None
    with local_debug_path.open('w', encoding='utf-8', errors='replace') as f:
        for line in proc.stdout:
            f.write(line)
            if should_live_print(line):
                print(line, end='', flush=True)
            if 'Best metric:' in line:
                mirror_checkpoints_to_drive(work_name)
    proc.wait()

    elapsed = (time.time() - start) / 60
    run_dir = latest_run_dir(work_name)
    copy_debug_log(local_debug_path, drive_debug_path)
    mirror_checkpoints_to_drive(work_name)

    if proc.returncode != 0:
        output_tail = read_tail(local_debug_path, 12000)
        print('\n[error tail from subprocess]')
        print(output_tail)
        print('\n[error tail from recorder log]')
        if run_dir:
            print(tail_log(run_dir, 120))
        else:
            print('(no run_dir/log was created before the crash)')
        print('local debug log:', local_debug_path)
        print('drive debug log:', drive_debug_path)
        raise subprocess.CalledProcessError(proc.returncode, cmd, output=output_tail)

    print(f'\n[done] {work_name}, elapsed_min={elapsed:.2f}')
    print('run_dir:', run_dir)
    print('local debug log:', local_debug_path)
    print('drive debug log:', drive_debug_path)
    return run_dir

smoke_dir = run_training(CONFIG_SMOKE, f'{RUN_PREFIX}_smoke', enabled=RUN_SMOKE)


In [ ]:
if smoke_dir:
    print(tail_log(smoke_dir, 120))
    ckpts = sorted((smoke_dir / 'ckpt').glob('*.pth')) if (smoke_dir / 'ckpt').exists() else []
    print('checkpoints:', [p.name for p in ckpts])
    assert ckpts, 'smoke run did not save checkpoint'
    print('STOP CHECK C passed: smoke checkpoint exists.')


## 6. Full Fine-Tuning

Full run은 `ResNet18_CULane.pth`에서 시작한다. v1 기준 실험이므로 우선 teacher/KD 없이 supervised fine-tuning만 수행한다.

현재 설정:

- epochs: 12
- batch size: 8
- lr: 1e-4
- scheduler: CosineAnnealingLR
- eval/save interval: 2 epoch

결과가 충분하지 않으면 다음 실험에서 데이터 정책, split, augmentation, teacher/KD를 별도로 다룬다.


In [ ]:
full_dir = run_training(CONFIG_FULL, f'{RUN_PREFIX}_full', enabled=RUN_FULL)


In [ ]:
if full_dir:
    print(tail_log(full_dir, 160))
    ckpt_dir = full_dir / 'ckpt'
    ckpts = sorted(ckpt_dir.glob('*.pth')) if ckpt_dir.exists() else []
    print('checkpoints:', [p.name for p in ckpts])
    assert ckpts, 'full run did not save checkpoint'
    print('best exists:', (ckpt_dir / 'best.pth').exists())


## 7. 결과 Drive 보관

Drive에는 전체 work_dir, 생성 config, 데이터셋 summary를 함께 저장한다. 이후 로컬로 내려받아 분석 노트북에서 비교한다.


In [ ]:
def copy_tree_to_drive(src: Path, dst: Path):
    if not src or not src.exists():
        print('missing src, skip:', src)
        return None
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print('copied:', src, '->', dst)
    return dst

run_dirs = []
if smoke_dir:
    run_dirs.append(smoke_dir)
if full_dir:
    run_dirs.append(full_dir)

for rd in run_dirs:
    copy_tree_to_drive(rd, OUT_DRIVE / rd.parent.name / rd.name)

meta_dir = OUT_DRIVE / 'meta'
meta_dir.mkdir(parents=True, exist_ok=True)
for p in [CONFIG_SMOKE, CONFIG_FULL, DATASET_DIR / 'build_summary.json', DATASET_DIR / 'validation_summary.json']:
    shutil.copy2(p, meta_dir / p.name)

manifest = {
    'drive_dir': str(DRIVE_DIR),
    'repo_dir': str(REPO_DIR),
    'dataset_name': DATASET_NAME,
    'pretrained': PRETRAINED_NAME,
    'run_prefix': RUN_PREFIX,
    'smoke_dir': str(smoke_dir) if smoke_dir else None,
    'full_dir': str(full_dir) if full_dir else None,
    'train_rows': train_count,
    'val_rows': val_count,
    'test_rows': test_count,
    'input_contract': 'BGR float32 CHW, range [0,1], raw crop cut_height=445 then Resize 800x320',
}
(meta_dir / 'run_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(manifest, ensure_ascii=False, indent=2))


In [ ]:
# Optional: make a compact archive of copied outputs on Drive.
archive_base = OUT_DRIVE / 'localfit_v1_colab_outputs'
archive_path = shutil.make_archive(str(archive_base), 'gztar', root_dir=OUT_DRIVE)
print('archive:', archive_path)
print('done. Download this output folder or archive after the Colab run finishes.')


## 결과 해석 메모

이 노트북이 끝난 뒤 확인할 파일은 다음과 같다.

- `outputs_localfit_v1/MapLane_LocalFit_Field12_v1_full/.../ckpt/best.pth`
- `outputs_localfit_v1/MapLane_LocalFit_Field12_v1_full/.../log.txt`
- `outputs_localfit_v1/meta/run_manifest.json`

다음 노트북에서는 `best.pth`를 로컬에서 분석하고, 반드시 학습 입력과 동일한 `/255` 전처리로 ONNX export/runtime을 구성한다.
